# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing all data elements by their Croissant `@id` fields for reproducibility and transparency.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset identifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their Croissant `@id` fields.

In [ ]:
# List all record sets with their @id and all their fields' @id
record_set_ids = []
print("Available record sets and their fields:\n")
for record_set in dataset.record_sets:
    print(f"RecordSet @id: {record_set['@id']}")
    record_set_ids.append(record_set['@id'])
    if 'field' in record_set:
        for field in record_set['field']:
            print(f"  |- Field @id: {field['@id']}  [name: {field.get('name', '')}]")
    print()
if not record_set_ids:
    print("No record sets discovered by schema. The dataset may load default table(s) instead.")

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames for analysis. Use the record set and field `@id`s discovered above.

In [ ]:
dataframes = {}
if record_set_ids:
    # Use discovered record sets
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet @id: {record_set_id}")
    # Print columns of the first discovered record set
    first_rs = record_set_ids[0]
    print(f"\nFields/Columns in RecordSet @id '{first_rs}':")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    # Fallback: try loading main tabular data (if the schema didn't enumerate record sets)
    print("No explicit record sets in schema. Attempting to load main data file.")
    main_df = pd.DataFrame(list(dataset.records()))
    print(f"Loaded {len(main_df)} records from dataset.")
    print(main_df.columns.tolist())
    display(main_df.head())
    dataframes['default'] = main_df

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filter records by a numeric field, normalize it, and group by a categorical field. All field references use their Croissant `@id`.

In [ ]:
# If specific record set(s) and fields are known use their @id here, else pick defaults from loaded DataFrame
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    # List columns for manual selection
    print(f"Columns in record set '{record_set_id}': {df.columns.tolist()}")
    # Try to pick likely numeric and group fields as examples
    # Example field IDs (replace with correct @id from data if necessary)
    numeric_candidate_fields = [col for col in df.columns if 'log_likelihood' in col.lower() or 'coefficient' in col.lower() or 'value' in col.lower() or df[col].dtype in [np.float64, np.int64]]
    group_candidate_fields = [col for col in df.columns if 'gender' in col.lower() or 'ward' in col.lower() or 'county' in col.lower() or df[col].dtype == object]
    # Pick first candidates or fallback
    numeric_field_id = numeric_candidate_fields[0] if numeric_candidate_fields else df.columns[0]
    group_field_id = group_candidate_fields[0] if group_candidate_fields else None

    print(f"Selected numeric field for analysis: '{numeric_field_id}' (@id)")
    if group_field_id:
        print(f"Selected group field: '{group_field_id}' (@id)")

    # Clean numeric (missing, non-numeric)
    df_clean_numeric = pd.to_numeric(df[numeric_field_id], errors='coerce')
    df_valid = df.copy()
    df_valid[numeric_field_id] = df_clean_numeric
    threshold = df_clean_numeric.mean() if not np.isnan(df_clean_numeric.mean()) else 0.0
    filtered_df = df_valid[df_valid[numeric_field_id] > threshold]

    print(f"Filtered records with '{numeric_field_id}' (@id) > {threshold:.3f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    mu, sigma = filtered_df[numeric_field_id].mean(), filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mu) / sigma
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the group field, if present
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)[[numeric_field_id]]
        print(f"Grouped data by '{group_field_id}' (@id):")
        display(grouped_df.head())
else:
    print("No dataframes available for analysis.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset using their Croissant `@id`.

In [ ]:
# Example: Histogram of numeric field, boxplot by group field if available
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    plt.figure(figsize=(8,4))
    plt.hist(df[numeric_field_id].dropna(), bins=20, alpha=0.75, color='skyblue')
    plt.xlabel(f"{numeric_field_id} (@id)")
    plt.ylabel("Count")
    plt.title(f"Distribution of '{numeric_field_id}' (@id)")
    plt.show()
    if group_field_id:
        plt.figure(figsize=(8,5))
        df.boxplot(column=numeric_field_id, by=group_field_id, grid=False)
        plt.title(f"Boxplot of '{numeric_field_id}' by '{group_field_id}' (@id)")
        plt.suptitle("")
        plt.xlabel(f"{group_field_id} (@id)")
        plt.ylabel(f"{numeric_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and process a Croissant-structured dataset using the `mlcroissant` library, referencing all entities by their persistent `@id` fields. We loaded the FAIR² Northern Kenya rangeland dataset, inspected its structure, filtered and normalized a numeric field by `@id`, and visualized key relationships by data groupings. This workflow supports reproducible and FAIR machine learning research.